In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from sklearn.decomposition import PCA

In [100]:
# Loading
datefruit = pd.read_csv("datasets/datefruit.csv")
datefruit_df = datefruit.copy()

In [101]:
# Inspecting 
datefruit_df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [102]:
datefruit_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 898 entries, 0 to 897
Data columns (total 35 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   AREA           898 non-null    int64  
 1   PERIMETER      898 non-null    float64
 2   MAJOR_AXIS     898 non-null    float64
 3   MINOR_AXIS     898 non-null    float64
 4   ECCENTRICITY   898 non-null    float64
 5   EQDIASQ        898 non-null    float64
 6   SOLIDITY       898 non-null    float64
 7   CONVEX_AREA    898 non-null    int64  
 8   EXTENT         898 non-null    float64
 9   ASPECT_RATIO   898 non-null    float64
 10  ROUNDNESS      898 non-null    float64
 11  COMPACTNESS    898 non-null    float64
 12  SHAPEFACTOR_1  898 non-null    float64
 13  SHAPEFACTOR_2  898 non-null    float64
 14  SHAPEFACTOR_3  898 non-null    float64
 15  SHAPEFACTOR_4  898 non-null    float64
 16  MeanRR         898 non-null    float64
 17  MeanRG         898 non-null    float64
 18  MeanRB         898 no

In [103]:
# Duplication
datefruit_df.duplicated().sum()

np.int64(0)

In [104]:
# Inconsistency
# No inconsistency for input features (no string feautres)
datefruit_df["Class"].unique()

<StringArray>
['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY']
Length: 7, dtype: str

In [105]:
# Missing values
datefruit_df.isnull().sum()
# No null values detected

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [106]:
# Split input and output
X = datefruit_df.drop(columns=["Class"])
y = datefruit_df["Class"]

In [107]:
# Split train and test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

In [108]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(
   y_train 
)
y_test_encoded = le.transform(
    y_test
)

In [109]:
# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(
    X_train
)
X_test_scaled = scaler.transform(
    X_test
)

# Deep Learning

In [110]:
# Conversion to tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.long)

In [111]:
# Creating Dataset and DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [112]:
# Build Model
class DateFruitANN(nn.Module):

    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 32),
            nn.ReLU(),

            nn.Linear(32, 32),
            nn.ReLU(),

            nn.Linear(32, len(y.unique()))
        )

    def forward(self, x):
        return self.model(x)

In [113]:
# Create model
ann_model = DateFruitANN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(ann_model.parameters())


In [114]:
# Training
 
epochs = 100
train_losses = []

for epoch in range(0, epochs):
    ann_model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad() # clearing previous gradients
        y_pred = ann_model(xb) # predicting output
        loss = criterion(y_pred, yb) # loss calculate
        loss.backward() # calculate gradients
        optimizer.step() # parameters are adjusted

        train_loss += loss.item()

    curr_train_loss = train_loss / len(train_loader)
    train_losses.append(curr_train_loss)
    print(f"Epoch: {epoch} => {curr_train_loss}")



Epoch: 0 => 1.8275236379016528
Epoch: 1 => 1.4966470084407113
Epoch: 2 => 1.099978814070875
Epoch: 3 => 0.9160385727882385
Epoch: 4 => 0.6756504438817501
Epoch: 5 => 0.5689373991706155
Epoch: 6 => 0.5544641275297512
Epoch: 7 => 0.44832258739254693
Epoch: 8 => 0.4435249377380718
Epoch: 9 => 0.3829535963860425
Epoch: 10 => 0.355849488215013
Epoch: 11 => 0.3269526350565932
Epoch: 12 => 0.30720429340462113
Epoch: 13 => 0.2892304367851466
Epoch: 14 => 0.2884804484519092
Epoch: 15 => 0.2699414146217433
Epoch: 16 => 0.2943556247787042
Epoch: 17 => 0.25229304148392245
Epoch: 18 => 0.23171599747464908
Epoch: 19 => 0.2228170120136135
Epoch: 20 => 0.21803366570648822
Epoch: 21 => 0.20731435526183553
Epoch: 22 => 0.20148339799859308
Epoch: 23 => 0.19400609893686618
Epoch: 24 => 0.2026375304399566
Epoch: 25 => 0.21163061701438643
Epoch: 26 => 0.22952634214677595
Epoch: 27 => 0.20379359510579062
Epoch: 28 => 0.17366259104681542
Epoch: 29 => 0.16844859846274962
Epoch: 30 => 0.1664392019801281
Epoch: 

In [115]:
# Evaluation
ann_model.eval()

y_actual = []
y_pred = []

with torch.no_grad():
    for xb, yb in test_loader:
        y_pred_probs = ann_model(xb)
        _, y_pred_labels = torch.max(y_pred_probs, 1)

        y_actual.extend(yb.cpu().numpy())
        y_pred.extend(y_pred_labels.cpu().numpy())

        

print(y_actual)
print(y_pred)

[np.int64(0), np.int64(6), np.int64(2), np.int64(6), np.int64(2), np.int64(4), np.int64(5), np.int64(4), np.int64(0), np.int64(5), np.int64(4), np.int64(2), np.int64(5), np.int64(6), np.int64(5), np.int64(2), np.int64(2), np.int64(2), np.int64(4), np.int64(5), np.int64(4), np.int64(4), np.int64(4), np.int64(3), np.int64(5), np.int64(6), np.int64(5), np.int64(3), np.int64(5), np.int64(5), np.int64(5), np.int64(6), np.int64(4), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(4), np.int64(4), np.int64(0), np.int64(2), np.int64(6), np.int64(2), np.int64(5), np.int64(1), np.int64(2), np.int64(6), np.int64(4), np.int64(4), np.int64(4), np.int64(2), np.int64(2), np.int64(5), np.int64(2), np.int64(0), np.int64(4), np.int64(3), np.int64(5), np.int64(2), np.int64(4), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(6), np.int64(5), np.int64(5), np.int64(3), np.int64(1), np.int64(2), np.int64(6), np.int64(1), np.int64(6), np.int64(0), np.int64(2), np.int64(6), np.int64(5)

In [116]:
print(f"Accuracy: {accuracy_score(y_actual, y_pred)}")
print(classification_report(y_actual, y_pred))

Accuracy: 0.9244444444444444
              precision    recall  f1-score   support

           0       0.88      0.93      0.90        15
           1       0.78      0.88      0.82        24
           2       1.00      0.97      0.98        60
           3       0.71      0.92      0.80        13
           4       1.00      0.89      0.94        45
           5       1.00      0.98      0.99        42
           6       0.85      0.85      0.85        26

    accuracy                           0.92       225
   macro avg       0.89      0.92      0.90       225
weighted avg       0.93      0.92      0.93       225



# Machine Learning

In [117]:
# Logistic Regression
lr_model = LogisticRegression()
lr_model.fit(X_train_scaled, y_train_encoded)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [118]:
# Decison tree
dt_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=5,
    random_state=42
)
dt_model.fit(X_train_scaled, y_train_encoded)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at 

In [119]:
# XGB
xgbc_model = XGBClassifier(max_depth=5)
xgbc_model.fit(X_train_scaled, y_train_encoded)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import l

In [120]:
# Evaluation
models = {
    "Logistic Regression": lr_model,
    "Decision tree": dt_model,
    "XGBoost": xgbc_model
}

for name, model in models.items():
    print(name + ":")
    y_pred = model.predict(X_test_scaled)
    print(f"Accuracy: {accuracy_score(y_test_encoded, y_pred)}")
    print(classification_report(y_test_encoded, y_pred))

Logistic Regression:
Accuracy: 0.9377777777777778
              precision    recall  f1-score   support

           0       0.93      0.93      0.93        15
           1       0.79      0.79      0.79        24
           2       0.98      0.97      0.97        60
           3       0.81      1.00      0.90        13
           4       1.00      0.91      0.95        45
           5       1.00      1.00      1.00        42
           6       0.86      0.92      0.89        26

    accuracy                           0.94       225
   macro avg       0.91      0.93      0.92       225
weighted avg       0.94      0.94      0.94       225

Decision tree:
Accuracy: 0.8222222222222222
              precision    recall  f1-score   support

           0       0.43      0.40      0.41        15
           1       0.62      0.67      0.64        24
           2       1.00      0.90      0.95        60
           3       0.57      1.00      0.72        13
           4       1.00      0.80     

ANN & Logistic Regression turns out to be the best model.

# Dimensionality reduction

In [121]:
pca = PCA(n_components=6, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
np.sum(pca.explained_variance_ratio_)

np.float64(0.9056986814746688)

In [122]:
# Conversion to tensors
X_train_tensor = torch.tensor(X_train_pca, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_pca, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.long)

In [123]:
# Creating Dataset and DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [124]:
X_train_pca.shape

(673, 6)

In [125]:
# Build Model
class DateFruitANN(nn.Module):

    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(X_train_pca.shape[1], 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, len(y.unique()))
        )

    def forward(self, x):
        return self.model(x)

In [126]:
# Create model
ann_model = DateFruitANN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(ann_model.parameters())


In [127]:
# Training
 
epochs = 100
train_losses = []

for epoch in range(0, epochs):
    ann_model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad() # clearing previous gradients
        y_pred = ann_model(xb) # predicting output
        loss = criterion(y_pred, yb) # loss calculate
        loss.backward() # calculate gradients
        optimizer.step() # parameters are adjusted

        train_loss += loss.item()

    curr_train_loss = train_loss / len(train_loader)
    train_losses.append(curr_train_loss)
    print(f"Epoch: {epoch} => {curr_train_loss}")



Epoch: 0 => 1.594574977051128
Epoch: 1 => 1.0013946972110055
Epoch: 2 => 0.7413583370772275
Epoch: 3 => 0.5679759541949765
Epoch: 4 => 0.5241564444520257
Epoch: 5 => 0.492089719934897
Epoch: 6 => 0.41928115859627724
Epoch: 7 => 0.38799175558167254
Epoch: 8 => 0.36804430616426875
Epoch: 9 => 0.3521607702555643
Epoch: 10 => 0.3403486650310118
Epoch: 11 => 0.38211176747625525
Epoch: 12 => 0.39177974923090503
Epoch: 13 => 0.3519348190589385
Epoch: 14 => 0.3251372386108745
Epoch: 15 => 0.32211856747215445
Epoch: 16 => 0.32387765564701776
Epoch: 17 => 0.29350484613413835
Epoch: 18 => 0.28995505517179315
Epoch: 19 => 0.3611058525063775
Epoch: 20 => 0.29281947750397114
Epoch: 21 => 0.2728959999301217
Epoch: 22 => 0.2798856401985342
Epoch: 23 => 0.2726126817817038
Epoch: 24 => 0.2650830795239678
Epoch: 25 => 0.25189502494917676
Epoch: 26 => 0.24993458627299828
Epoch: 27 => 0.2604462765157223
Epoch: 28 => 0.25003483197228477
Epoch: 29 => 0.25689986043355684
Epoch: 30 => 0.2631401504305276
Epoch:

In [128]:
# Evaluation
ann_model.eval()

y_actual = []
y_pred = []

with torch.no_grad():
    for xb, yb in test_loader:
        y_pred_probs = ann_model(xb)
        _, y_pred_labels = torch.max(y_pred_probs, 1)

        y_actual.extend(yb.cpu().numpy())
        y_pred.extend(y_pred_labels.cpu().numpy())

        

print(y_actual)
print(y_pred)

[np.int64(4), np.int64(3), np.int64(6), np.int64(5), np.int64(4), np.int64(5), np.int64(2), np.int64(4), np.int64(6), np.int64(2), np.int64(6), np.int64(3), np.int64(6), np.int64(5), np.int64(2), np.int64(5), np.int64(5), np.int64(6), np.int64(2), np.int64(2), np.int64(2), np.int64(6), np.int64(2), np.int64(2), np.int64(4), np.int64(3), np.int64(4), np.int64(1), np.int64(2), np.int64(6), np.int64(1), np.int64(1), np.int64(2), np.int64(4), np.int64(5), np.int64(2), np.int64(2), np.int64(4), np.int64(5), np.int64(0), np.int64(2), np.int64(4), np.int64(4), np.int64(3), np.int64(6), np.int64(6), np.int64(4), np.int64(2), np.int64(0), np.int64(1), np.int64(5), np.int64(4), np.int64(3), np.int64(2), np.int64(2), np.int64(5), np.int64(2), np.int64(4), np.int64(2), np.int64(3), np.int64(5), np.int64(5), np.int64(4), np.int64(2), np.int64(5), np.int64(1), np.int64(1), np.int64(5), np.int64(6), np.int64(4), np.int64(2), np.int64(2), np.int64(0), np.int64(3), np.int64(1), np.int64(2), np.int64(6)

In [129]:
print(f"Accuracy: {accuracy_score(y_actual, y_pred)}")
print(classification_report(y_actual, y_pred))

Accuracy: 0.8933333333333333
              precision    recall  f1-score   support

           0       0.85      0.73      0.79        15
           1       0.66      0.88      0.75        24
           2       1.00      0.88      0.94        60
           3       0.68      1.00      0.81        13
           4       1.00      0.87      0.93        45
           5       1.00      1.00      1.00        42
           6       0.81      0.85      0.83        26

    accuracy                           0.89       225
   macro avg       0.86      0.89      0.86       225
weighted avg       0.91      0.89      0.90       225

